# 09. Tradeoffs and Final Project Summary

This notebook summarizes the main tradeoffs in the CTR prediction project and provides a complete final explanation of the problem framing, dataset design, feature engineering, model selection, training strategy, evaluation metrics, and limitations.

This assignment focuses only on offline model development and evaluation. It does not cover deployment, online serving, monitoring, retraining, infrastructure, or system architecture.

## Project Summary Flow

```mermaid
flowchart TD
    A[Problem Framing] --> B[Dataset Understanding]
    B --> C[Time-Based Data Split]
    C --> D[Feature Engineering]
    D --> E[Baseline Model]
    E --> F[Advanced Model Comparison]
    F --> G[Evaluation Metrics]
    G --> H[Model Interpretation]
    H --> I[Tradeoffs and Limitations]
    I --> J[Final Model Choice]
```


## 1. Problem Framing

The goal of this project is to predict whether an advertisement impression will result in a click.

The target variable is binary:

- `0` means no click
- `1` means click

The model predicts a click probability rather than only a click or no-click label.

For example:

- `0.10` means a 10% predicted probability of a click
- `0.70` means a 70% predicted probability of a click

Because the task is probability prediction, Log Loss is used as the main evaluation metric.

## 2. Dataset Design

The Avazu CTR dataset contains more than 40 million advertisement-impression records.

Each row represents one advertisement impression and includes:

- Time information
- Site and application information
- Device information
- Banner position
- Anonymous categorical fields such as `C14`, `C17`, and `C20`
- The click target

The dataset was sorted by time and divided into:

- 70% training data
- 15% validation data
- 15% test data

A time-based split was used instead of a random split so that the model is trained on earlier records and evaluated on later records. This better represents real-world prediction and reduces future-data leakage.

## 3. Feature Engineering Approach

The following feature-engineering steps were used:

### Time features

- Day
- Hour of day
- Day of week
- Weekend indicator
- Time period

### Frequency encoding

High-cardinality categorical features were converted into frequency values.

Examples include:

- `device_id_freq`
- `device_ip_freq`
- `app_id_freq`
- `site_id_freq`
- `site_domain_freq`
- `C14_freq`
- `C17_freq`

The frequency mappings were learned only from the training data and then applied to validation and test data.

Unseen validation or test categories were assigned a value of `0`.

This approach reduced memory usage and avoided creating millions of one-hot encoded columns.

## 4. Model Choices

### Logistic Regression

Logistic Regression was selected as the baseline model because it is simple, fast, and directly predicts click probabilities.

It provides a reference point for deciding whether more complex models provide meaningful improvement.

### HistGradientBoosting

HistGradientBoosting was selected as the main advanced model because it can learn nonlinear patterns and feature interactions.

It is also more efficient than traditional Gradient Boosting for large tabular datasets.

### Other models considered

- Decision Tree
- Random Forest


## 5. Training Strategy

The full training dataset contained more than 28 million rows and caused local memory limitations.

Therefore, the initial model comparison used:

- 500,000 training records
- 100,000 validation records

The samples were taken only after the time-based split.

This preserved the separation between training and validation data.

Logistic Regression used scaled features, while HistGradientBoosting used the original numerical values because tree-based models do not require feature scaling.

## 6. Evaluation Metrics

The models were evaluated using:

- **Log Loss:** Measures the quality of predicted click probabilities. Lower is better.
- **ROC-AUC:** Measures how well the model ranks clicks above non-clicks. Higher is better.
- **PR-AUC:** Focuses on the less common click class. Higher is better.
- **Precision:** Of the predicted clicks, how many were actual clicks.
- **Recall:** Of the actual clicks, how many the model identified.
- **F1 Score:** Balances precision and recall.
- **Calibration:** Checks whether predicted probabilities match actual click rates.
- **Lift:** Measures whether the model identifies higher-CTR groups than random selection.

Accuracy was not used as the main metric because most impressions are non-clicks. A model could predict no click for almost every row and still achieve high accuracy.

In [1]:
import pandas as pd

model_comparison = pd.DataFrame({
    "Model": [
        "Logistic Regression",
        "HistGradientBoosting"
    ],
    "Log_Loss": [
        0.399155,
        0.374304
    ],
    "ROC_AUC": [
        0.621859,
        0.658727
    ]
})

model_comparison

,Model,Log_Loss,ROC_AUC
0,Logistic Regression,0.399155,0.621859
1,HistGradientBoosting,0.374304,0.658727


## 7. Model-Performance Interpretation

HistGradientBoosting performed better than Logistic Regression.

- Log Loss improved from `0.399155` to `0.374304`
- ROC-AUC improved from `0.621859` to `0.658727`

This suggests that the CTR problem contains nonlinear relationships and feature interactions that Logistic Regression cannot capture as effectively.

HistGradientBoosting is therefore the best-performing model among the models tested.

# 8. Tradeoffs


## 8.1 Accuracy vs Interpretability

More accurate models are often harder to explain.

Logistic Regression is easier to interpret because each feature has a coefficient showing whether it increases or decreases the predicted click probability.

Gradient Boosting can produce better predictions because it learns nonlinear rules and interactions, but it is more difficult to explain directly.

The tradeoff is:

- Simple model → easier to explain, but may miss complex patterns
- Complex model → better predictive performance, but harder to interpret

For this project, HistGradientBoosting was selected because the improvement in Log Loss and ROC-AUC was meaningful. Permutation importance and segment analysis were used to improve interpretability.

## 8.2 Simple Models versus Complex Models

Simple models such as Logistic Regression:

- Train faster
- Use less memory
- Are easier to debug
- Are easier to explain
- Provide a strong baseline

Complex models such as Gradient Boosting, Factorization Machines, and neural networks:

- Learn nonlinear relationships
- Capture feature interactions
- May provide better predictive performance
- Require more tuning
- Require more computation
- Are harder to interpret

A complex model should be selected only when it produces a meaningful improvement over the baseline.

## 8.3 Risks of Historical Click Behavior

Historical click behavior can be useful because it captures past user, device, app, and site activity.

However, it creates several risks:

### Popularity bias

Popular ads or apps may receive more clicks simply because they were shown more often.

The model may incorrectly learn that high exposure always means high relevance.

### Feedback loops

If popular ads are shown more often, they collect more clicks. The model may then rank them even higher, causing them to receive even more exposure.

### New-item disadvantage

New or rare ads, apps, and sites have little historical data. The model may assign them low scores even when they are relevant.

### Data leakage

Historical features must be calculated only from information available before the prediction time. Using future click information would make validation results unrealistically strong.

### Changing behavior

Past click behavior may not represent future behavior because user interests, campaigns, devices, and ad inventory can change.

## 8.4 Avoiding a Popularity-Only Model

The model should learn true user-ad relevance rather than only popularity.

The following approaches help reduce popularity bias:

### Use multiple feature groups

Do not rely only on frequency features. Include time, device, placement, site, app, category, and context features.

### Evaluate rare and popular groups separately

Compare model performance across low-, medium-, and high-frequency devices, apps, ads, and sites.

If performance is much better for popular groups, the model may not generalize well to rare groups.

### Use time-based validation

Time-based validation tests whether patterns learned from past data remain useful for later records.

### Limit or transform frequency features

Very large frequency values can dominate the model. Log transformation, clipping, or frequency bands can reduce this effect.

### Add interaction features

Interactions such as device and app, site and banner position, or time and category can represent relevance more directly than popularity alone.

### Compare calibration and lift by frequency group

The model should produce useful probabilities and lift for rare as well as popular groups.

### Keep anonymous-column limitations clear

Anonymous Avazu fields should be treated as proxies unless their meaning is confirmed.

## 9. Final Model Choice

HistGradientBoosting was selected as the best model among the tested models because:

- It achieved lower Log Loss than Logistic Regression
- It achieved higher ROC-AUC than Logistic Regression
- It captured nonlinear relationships
- It captured feature interactions
- It was computationally practical for the sampled tabular dataset

The selected classification threshold was `0.15` because it produced the highest F1 score among the thresholds tested.

The threshold can be changed depending on the business goal:

- Lower threshold → higher recall, lower precision
- Higher threshold → higher precision, lower recall

## 10. Final Assignment Answer

The project was framed as a binary CTR prediction problem where each advertisement impression is classified as click or no click, while the model also produces a click probability.

The Avazu dataset contained more than 40 million time-ordered impressions. A 70% training, 15% validation, and 15% test split was created after sorting by timestamp. This reduced future-data leakage and represented real-world prediction more accurately than a random split.

Feature engineering included time-based features and frequency encoding for high-cardinality categorical columns. Frequency mappings were learned from training data only and then applied to validation and test data. This reduced memory usage and avoided very large one-hot encoded matrices.

Logistic Regression was used as the baseline because it is simple, fast, and directly predicts probabilities. HistGradientBoosting was then evaluated because it can capture nonlinear relationships and feature interactions. It improved Log Loss from 0.399155 to 0.374304 and ROC-AUC from 0.621859 to 0.658727.

The main evaluation metric was Log Loss because CTR prediction is a probability-estimation problem. ROC-AUC and PR-AUC were used to evaluate ranking performance, while precision, recall, F1 score, calibration, and lift were used as supporting metrics.

The main tradeoff was between model simplicity and predictive performance. Logistic Regression was easier to interpret, while HistGradientBoosting provided better performance but required interpretation tools such as permutation importance and segment analysis.

Historical click and frequency features created risks of popularity bias, feedback loops, and poor performance for rare or new entities. To reduce these risks, the model was evaluated across rare, medium, and popular groups, and frequency features were combined with contextual features rather than used alone.

Based on the offline validation results, HistGradientBoosting was selected as the best-performing model among the models tested.